# 🦙 LlamaIndex — RAG-First Framework

## LlamaIndex vs LangChain

Both are RAG frameworks. The main difference in focus:

```
LangChain:   General-purpose LLM orchestration (chains, agents, tools)
             → Great when RAG is one part of a larger app

LlamaIndex:  Specialised in data indexing and retrieval
             → Great when RAG IS the app
```

LlamaIndex has more retrieval strategies out-of-the-box: tree index, keyword index, SQL, knowledge graphs, etc.

## Core Concepts

```
Documents  →  Nodes (chunks with metadata)
                ↓
           VectorStoreIndex  ← embeds everything
                ↓
           QueryEngine       ← search + generate
```

In [1]:
# !pip install llama-index llama-index-embeddings-huggingface

In [2]:
# Step 1: Create documents
from llama_index.core import Document

documents = [
    Document(text="RAG retrieves relevant documents and passes them as context to an LLM."),
    Document(text="Chunking divides documents into smaller segments to fit within LLM context limits."),
    Document(text="Vector stores index embeddings for fast nearest-neighbour search."),
    Document(text="Reranking improves precision by re-scoring top-k retrieved documents."),
    Document(text="Evaluation metrics like faithfulness and relevance measure RAG quality."),
]

print(f"Created {len(documents)} documents")

Created 5 documents


In [3]:
# Step 2: Build an index using a local embedding model (no API key needed)
from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Use a local embedding model instead of OpenAI
Settings.embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")
Settings.llm = None  # We'll handle LLM calls manually for now

# Build the index — this embeds all documents
index = VectorStoreIndex.from_documents(documents)
print("Index built!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

LLM is explicitly disabled. Using MockLLM.
Index built!


In [4]:
# Step 3: Retrieve (without LLM generation)
retriever = index.as_retriever(similarity_top_k=2)

question = "How does RAG work?"
nodes = retriever.retrieve(question)

print(f"Query: '{question}'\n")
for i, node in enumerate(nodes, 1):
    print(f"  {i}. [score: {node.score:.3f}] {node.text}")

Query: 'How does RAG work?'

  1. [score: 0.483] Evaluation metrics like faithfulness and relevance measure RAG quality.
  2. [score: 0.418] RAG retrieves relevant documents and passes them as context to an LLM.


In [5]:
# Step 4: Full query engine with LLM (Anthropic)

llamaindex_with_llm = """
from llama_index.llms.anthropic import Anthropic
from llama_index.core import Settings

# Set the LLM
Settings.llm = Anthropic(model="claude-haiku-4-5-20251001")

# Create a query engine (retrieves + generates)
query_engine = index.as_query_engine()

# Ask a question
response = query_engine.query("What is chunking?")
print(response)                    # The answer
print(response.source_nodes)       # Retrieved docs used
"""

print("LlamaIndex with Anthropic LLM:")
print(llamaindex_with_llm)

LlamaIndex with Anthropic LLM:

from llama_index.llms.anthropic import Anthropic
from llama_index.core import Settings

# Set the LLM
Settings.llm = Anthropic(model="claude-haiku-4-5-20251001")

# Create a query engine (retrieves + generates)
query_engine = index.as_query_engine()

# Ask a question
response = query_engine.query("What is chunking?")
print(response)                    # The answer
print(response.source_nodes)       # Retrieved docs used



In [6]:
# LlamaIndex extras — features not in LangChain out-of-the-box

extras = """
# 1. SummaryIndex — summarise ALL docs, good for "give me an overview"
from llama_index.core import SummaryIndex
summary_index = SummaryIndex.from_documents(documents)

# 2. Keyword Index — fast keyword-based retrieval
from llama_index.core import SimpleKeywordTableIndex
kw_index = SimpleKeywordTableIndex.from_documents(documents)

# 3. Router — automatically picks the right index based on query type
from llama_index.core.query_engine import RouterQueryEngine
router = RouterQueryEngine.from_defaults(
    query_engine_tools=[vector_tool, summary_tool]
)
# Detailed question → routes to vector index
# "Give overview" → routes to summary index
"""

print("LlamaIndex power features:")
print(extras)

LlamaIndex power features:

# 1. SummaryIndex — summarise ALL docs, good for "give me an overview"
from llama_index.core import SummaryIndex
summary_index = SummaryIndex.from_documents(documents)

# 2. Keyword Index — fast keyword-based retrieval
from llama_index.core import SimpleKeywordTableIndex
kw_index = SimpleKeywordTableIndex.from_documents(documents)

# 3. Router — automatically picks the right index based on query type
from llama_index.core.query_engine import RouterQueryEngine
router = RouterQueryEngine.from_defaults(
    query_engine_tools=[vector_tool, summary_tool]
)
# Detailed question → routes to vector index
# "Give overview" → routes to summary index



## LangChain vs LlamaIndex — Quick Decision

| | LangChain | LlamaIndex |
|---|---|---|
| **Best for** | Complex agents, tool use | Deep RAG, multiple index types |
| **Learning curve** | Steeper | More focused |
| **Community** | Larger | Growing fast |
| **RAG out-of-box** | Good | Excellent |

**Honest advice:** Start with the from-scratch approach (this course), then pick whichever framework fits your use case. You'll understand both better once you know the fundamentals.

**Docs:** https://docs.llamaindex.ai